In [ ]:
import numpy as np
path = "/root/work/dhn/results/sinpend_kernel2_stride1/gen_sequence/denoise1_update1/result_dict.npy"
result=np.load(path, allow_pickle=True).item()

In [ ]:
p_scale=result["p_scale"]
t=result["t"]
q_gt=result["q_gt"]
q_pred=result["q_pred"]
t.shape, q_gt.shape, q_pred.shape,p_scale.shape, result.keys()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Change these to your actual gen folder
gen_dir = "/root/work/dhn/results/sinpend_kernel2_stride1/gen_sequence"
# If there are multiple gen configs inside, pick one
# e.g., "ns20_stepsize1" — list directories to see what's there
subdirs = [d for d in os.listdir(gen_dir) if os.path.isdir(os.path.join(gen_dir, d))]
assert subdirs, f"No subdirectories found in {gen_dir}"
cfg_dir = os.path.join(gen_dir, subdirs[0])

result = np.load(os.path.join(cfg_dir, "result_dict.npy"), allow_pickle=True).item()

t = result["t"]          # shape: [N, T, 1]
q_gt = result["q_gt"]    # shape: [N, T, D]
q_pred = result["q_pred"]# shape: [N, T, D]

p_gt=result["p_gt"]
p_pred=result["p_pred"]

i = 0      # sample index to visualize
d = 0      # dimension index (0 for single pendulum)

tt = t[i]
plt.figure(figsize=(8,5))
plt.plot(tt, q_gt[i,:,d], label="q_gt", lw=2)
plt.plot(tt, q_pred[i,:,d], label="q_pred", lw=2, alpha=0.8)
plt.xlabel("time")
plt.ylabel("q")
plt.title(f"Sample {i}, dim {d}")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(tt, p_gt[i,:,d], label="p_gt", lw=2)
plt.plot(tt, p_pred[i,:,d], label="p_pred", lw=2, alpha=0.8)
plt.xlabel("time")
plt.ylabel("p")
plt.title(f"Sample {i}, dim {d}")
plt.legend()

## Visualize the results for the two body problem

In [ ]:
# path = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise1_update1/result_dict.npy'
path = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise5_update1/result_dict.npy'
path_cond = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise5_update1/cond_dict.npy'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

data = np.load(path, allow_pickle=True).item()
cond = np.load(path_cond, allow_pickle=True).item()
print(data.keys())
print(cond.keys())

In [ ]:
data['t'].shape, data['q_gt'].shape, data['q_pred'].shape, data['p_gt'].shape, data['p_pred'].shape

In [ ]:
data['t'].shape

In [ ]:
cond['m1'].shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def visualize(data, traj_index, num_points = 20):
    if num_points is None:
        num_points = data['t'].shape[1]
    ##################################################
    t = data['t'][traj_index][:num_points]
    q_gt = data['q_gt'][traj_index][:num_points]
    q_pred = data['q_pred'][traj_index][:num_points]
    p_gt = data['p_gt'][traj_index][:num_points]
    p_pred = data['p_pred'][traj_index][:num_points]

    r_gt = q_gt[:, 0]
    theta_gt = q_gt[:, 1]
    r_pred = q_pred[:, 0]
    theta_pred = q_pred[:, 1]

    dr_gt = p_gt[:, 0]
    dtheta_gt = p_gt[:, 1]
    dr_pred = p_pred[:, 0]
    dtheta_pred = p_pred[:, 1]

    # Wrap theta to [0, 2π] for visualization
    theta_wrapped = theta_gt % (2 * np.pi)
    theta_wrapped_pred = theta_pred % (2 * np.pi)

    m1 = cond['m1'][traj_index]
    m2 = cond['m2'][traj_index]
    G = 1
    r1 = cond['r1'][traj_index]
    r2 = cond['r2'][traj_index]

    ##################################################
    # Plot q-t curve (r and theta vs time)
    ##################################################
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, r_gt, label='r (radial distance) - gt')
    plt.plot(t, r_pred, label='r (radial distance) - pred')
    plt.plot(t, theta_wrapped, label='θ (angle) - gt')
    plt.plot(t, theta_wrapped_pred, label='θ (angle) - pred')
    plt.title(f'Positions vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Position [r: distance, θ: rad]')
    plt.legend()
    plt.tight_layout()
    plt.grid(True)
    plt.show()

    ##################################################
    # Plot dq-t curve (velocities vs time)
    ##################################################
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, dr_gt, label='dr/dt (radial velocity) - gt')
    plt.plot(t, dr_pred, label='dr/dt (radial velocity) - pred')
    plt.plot(t, dtheta_gt, label='dθ/dt (angular velocity) - gt')

    plt.plot(t, dtheta_pred, label='dθ/dt (angular velocity) - pred')
    plt.title(f'Velocities vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.legend()
    plt.tight_layout()
    plt.grid(True)
    plt.show()

    ##################################################
    # Plot phase space: r-dr and theta-dtheta
    ##################################################
    plt.figure(figsize=(10, 6))
    plt.plot(r_gt, dr_gt, label='r vs dr/dt - gt')
    plt.plot(r_pred, dr_pred, label='r vs dr/dt - pred')
    plt.plot(theta_wrapped, dtheta_gt, label='θ vs dθ/dt - gt')
    plt.plot(theta_wrapped_pred, dtheta_pred, label='θ vs dθ/dt - pred')
    plt.title(f'Phase Space')
    plt.xlabel('Position [r: distance, θ: rad]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    ##################################################
    # Plot orbit trajectory (2D orbit path)
    ##################################################
    # -----------------------------
    # 4) Orbit trajectories (2D)
    # -----------------------------
    # Convert to Cartesian for reduced mass particle
    x_rel_gt   = r_gt   * np.cos(theta_gt)
    y_rel_gt   = r_gt   * np.sin(theta_gt)
    x_rel_pred = r_pred * np.cos(theta_pred)
    y_rel_pred = r_pred * np.sin(theta_pred)

    # Positions of the two bodies relative to center of mass
    x1_gt = -m2 / M * x_rel_gt
    y1_gt = -m2 / M * y_rel_gt
    x2_gt =  m1 / M * x_rel_gt
    y2_gt =  m1 / M * y_rel_gt

    x1_pred = -m2 / M * x_rel_pred
    y1_pred = -m2 / M * y_rel_pred
    x2_pred =  m1 / M * x_rel_pred
    y2_pred =  m1 / M * y_rel_pred

    plt.figure(figsize=(8, 8), dpi=100)
    # GT
    plt.plot(x1_gt, y1_gt, '-',  label='Body 1 – GT',   color='C0', alpha=0.6)
    plt.plot(x2_gt, y2_gt, '-',  label='Body 2 – GT',   color='C0', alpha=0.6, linestyle='--')
    # Pred
    plt.plot(x1_pred, y1_pred, '-', label='Body 1 – pred', color='C1', alpha=0.6)
    plt.plot(x2_pred, y2_pred, '-', label='Body 2 – pred', color='C1', alpha=0.6, linestyle='--')

    # Start/end markers for GT
    plt.plot(x1_gt[0],  y1_gt[0],  'o', color='C0', label='Body 1 start (GT)')
    plt.plot(x2_gt[0],  y2_gt[0],  'o', color='C2', label='Body 2 start (GT)')
    plt.plot(x1_gt[-1], y1_gt[-1], 's', color='C0', label='Body 1 end (GT)')
    plt.plot(x2_gt[-1], y2_gt[-1], 's', color='C2', label='Body 2 end (GT)')

    plt.plot(0, 0, 'k+', markersize=12, label='Center of Mass', markeredgewidth=2)
    plt.title(f'Orbit Trajectories (GT vs Pred)\n m1={m1:.2f}, m2={m2:.2f}, G={G:.2f}')
    plt.xlabel('x position')
    plt.ylabel('y position')
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # 5) Animated orbit (optional)
    # -----------------------------
    if not make_animation:
        return

    # Use GT for animation (you can swap to pred or overlay if you want)
    x1 = x1_gt
    y1 = y1_gt
    x2 = x2_gt
    y2 = y2_gt

    r_max = np.max(np.sqrt(x1**2 + y1**2))
    plot_margin = 0.2 * r_max
    xlim = (-r_max - plot_margin, r_max + plot_margin)
    ylim = (-r_max - plot_margin, r_max + plot_margin)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_title(f'Two-Body Orbit Animation\nm1={m1:.2f}, m2={m2:.2f}, G={G:.2f}')
    ax.set_xlabel('x position')
    ax.set_ylabel('y position')
    ax.grid(True, alpha=0.3)

    # CoM marker
    ax.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='CoM')

    # Bodies
    body1, = ax.plot([], [], 'o', color='C0', markersize=8, label='Body 1')
    body2, = ax.plot([], [], 'o', color='C1', markersize=8, label='Body 2')

    # Traces
    trace1, = ax.plot([], [], '-', color='C0', alpha=0.4)
    trace2, = ax.plot([], [], '-', color='C1', alpha=0.4)

    # Connection line
    connection, = ax.plot([], [], '--', color='gray', alpha=0.5)

    # Time label
    time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                        va='top', fontsize=12,
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.legend(loc='upper right')

    trace1_x, trace1_y = [], []
    trace2_x, trace2_y = [], []

    def update(frame):
        idx = frame * t_per_frame
        if idx >= len(t):
            idx = len(t) - 1

        x1_t, y1_t = x1[idx], y1[idx]
        x2_t, y2_t = x2[idx], y2[idx]

        # Bodies
        body1.set_data([x1_t], [y1_t])
        body2.set_data([x2_t], [y2_t])

        # Connection
        connection.set_data([x1_t, x2_t], [y1_t, y2_t])

        # Traces
        trace1_x.append(x1_t)
        trace1_y.append(y1_t)
        trace2_x.append(x2_t)
        trace2_y.append(y2_t)
        trace1.set_data(trace1_x, trace1_y)
        trace2.set_data(trace2_x, trace2_y)

        # Time
        time_text.set_text(f'Time: {t[idx]:.2f} s')

        return body1, body2, trace1, trace2, connection, time_text

    num_frames = max(1, len(t) // t_per_frame)
    ani = FuncAnimation(fig, update, frames=num_frames, blit=True, interval=50)

    plt.close(fig)  # prevent duplicate static image
    display(HTML(ani.to_jshtml()))


In [ ]:
data['t'].shape

In [ ]:
visualize(data, 10, num_points=None)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def visualize(
    data,
    cond,
    traj_index=0,
    num_points=None,
    make_animation=True,
    t_per_frame=1,
):
    """
    Visualize one two-body trajectory (GT vs prediction).

    Parameters
    ----------
    data : dict
        Expected keys: 't', 'q_gt', 'q_pred', 'p_gt', 'p_pred',
        each shaped (n_traj, T, dim).
    cond : dict
        Expected keys: 'm1', 'm2', optionally others per trajectory.
    traj_index : int
        Index of the trajectory to visualize.
    num_points : int or None
        Number of time steps to plot (from the start). If None, use full length.
    make_animation : bool
        Whether to generate an inline orbit animation (GT).
    t_per_frame : int
        Number of time steps between animation frames.
    """

    # ---------------------------------------------------------
    # Slice trajectory
    # ---------------------------------------------------------
    t_full = data['t'][traj_index]
    if num_points is None:
        num_points = t_full.shape[0]

    t      = t_full[:num_points]
    q_gt   = data['q_gt'][traj_index][:num_points]
    q_pred = data['q_pred'][traj_index][:num_points]
    p_gt   = data['p_gt'][traj_index][:num_points]
    p_pred = data['p_pred'][traj_index][:num_points]

    r_gt,    theta_gt    = q_gt[:, 0], q_gt[:, 1]
    r_pred,  theta_pred  = q_pred[:, 0], q_pred[:, 1]
    dr_gt,   dtheta_gt   = p_gt[:, 0], p_gt[:, 1]
    dr_pred, dtheta_pred = p_pred[:, 0], p_pred[:, 1]

    # Wrap theta to [0, 2π] for visualization
    theta_wrapped_gt   = theta_gt   % (2 * np.pi)
    theta_wrapped_pred = theta_pred % (2 * np.pi)

    # ---------------------------------------------------------
    # Physical parameters
    # ---------------------------------------------------------
    m1 = float(cond['m1'][traj_index])
    m2 = float(cond['m2'][traj_index])
    M  = m1 + m2
    G  = 1.0  # or use cond['G'][traj_index] if you have it

    # ---------------------------------------------------------
    # 1) q–t curves (r, θ vs time)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, r_gt,              label='r (radial) – GT',   color='C0')
    plt.plot(t, r_pred,            label='r (radial) – pred', color='C1')
    plt.plot(t, theta_wrapped_gt,  label='θ (angle) – GT',    color='C0', linestyle='--')
    plt.plot(t, theta_wrapped_pred,label='θ (angle) – pred',  color='C1', linestyle='--')
    plt.title('Positions vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Position [r: distance, θ: rad]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 2) dq–t curves (velocities vs time)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, dr_gt,       label='dr/dt – GT',   color='C0')
    plt.plot(t, dr_pred,     label='dr/dt – pred', color='C1')
    plt.plot(t, dtheta_gt,   label='dθ/dt – GT',   color='C0', linestyle='--')
    plt.plot(t, dtheta_pred, label='dθ/dt – pred', color='C1', linestyle='--')
    plt.title('Velocities vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 3) Phase space: (r, dr) and (θ, dθ)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(r_gt,       dr_gt,       label='r vs dr/dt – GT',   color='C0', alpha=0.7)
    plt.plot(r_pred,     dr_pred,     label='r vs dr/dt – pred', color='C1', alpha=0.7)
    plt.plot(theta_wrapped_gt,   dtheta_gt,   label='θ vs dθ/dt – GT',   color='C0', linestyle='--', alpha=0.7)
    plt.plot(theta_wrapped_pred, dtheta_pred, label='θ vs dθ/dt – pred', color='C1', linestyle='--', alpha=0.7)
    plt.title('Phase Space')
    plt.xlabel('Position [r: distance, θ: rad]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 4) Orbit trajectories (2D, GT vs pred)
    # ---------------------------------------------------------
    # Reduced coordinate → Cartesian
    x_rel_gt   = r_gt   * np.cos(theta_gt)
    y_rel_gt   = r_gt   * np.sin(theta_gt)
    x_rel_pred = r_pred * np.cos(theta_pred)
    y_rel_pred = r_pred * np.sin(theta_pred)

    # Positions of bodies in CoM frame
    x1_gt = -m2 / M * x_rel_gt
    y1_gt = -m2 / M * y_rel_gt
    x2_gt =  m1 / M * x_rel_gt
    y2_gt =  m1 / M * y_rel_gt

    x1_pred = -m2 / M * x_rel_pred
    y1_pred = -m2 / M * y_rel_pred
    x2_pred =  m1 / M * x_rel_pred
    y2_pred =  m1 / M * y_rel_pred

    plt.figure(figsize=(8, 8), dpi=100)
    # GT
    plt.plot(x1_gt, y1_gt, '-',  label='Body 1 – GT',   color='C0', alpha=0.7)
    plt.plot(x2_gt, y2_gt, '--', label='Body 2 – GT',   color='C0', alpha=0.7)
    # Pred
    plt.plot(x1_pred, y1_pred, '-',  label='Body 1 – pred', color='C1', alpha=0.7)
    plt.plot(x2_pred, y2_pred, '--', label='Body 2 – pred', color='C1', alpha=0.7)

    # Start/end markers (GT)
    plt.plot(x1_gt[0],  y1_gt[0],  'o', color='C0', label='Body 1 start (GT)')
    plt.plot(x2_gt[0],  y2_gt[0],  'o', color='C2', label='Body 2 start (GT)')
    plt.plot(x1_gt[-1], y1_gt[-1], 's', color='C0', label='Body 1 end (GT)')
    plt.plot(x2_gt[-1], y2_gt[-1], 's', color='C2', label='Body 2 end (GT)')

    plt.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='Center of Mass')
    plt.title(f'Orbit Trajectories (GT vs Pred)\n m1={m1:.2f}, m2={m2:.2f}, G={G:.2f}')
    plt.xlabel('x position')
    plt.ylabel('y position')
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 5) Animated orbit (GT only, optional)
    # ---------------------------------------------------------
    if not make_animation:
        return

    x1 = x1_gt
    y1 = y1_gt
    x2 = x2_gt
    y2 = y2_gt
    # x1 = x1_pred
    # y1 = y1_pred
    # x2 = x2_pred
    # y2 = y2_pred

    r_max = np.max(np.sqrt(x1**2 + y1**2))
    plot_margin = 0.2 * r_max
    xlim = (-r_max - plot_margin, r_max + plot_margin)
    ylim = (-r_max - plot_margin, r_max + plot_margin)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_title(f'Two-Body Orbit Animation\nm1={m1:.2f}, m2={m2:.2f}, G={G:.2f}')
    ax.set_xlabel('x position')
    ax.set_ylabel('y position')
    ax.grid(True, alpha=0.3)

    # CoM marker
    ax.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='CoM')

    # Bodies
    body1, = ax.plot([], [], 'o', color='C0', markersize=8, label='Body 1')
    body2, = ax.plot([], [], 'o', color='C1', markersize=8, label='Body 2')

    # Traces
    trace1, = ax.plot([], [], '-', color='C0', alpha=0.4)
    trace2, = ax.plot([], [], '-', color='C1', alpha=0.4)

    # Connection line
    connection, = ax.plot([], [], '--', color='gray', alpha=0.5)

    # Time label
    time_text = ax.text(
        0.02, 0.98, '', transform=ax.transAxes, va='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )

    ax.legend(loc='upper right')

    trace1_x, trace1_y = [], []
    trace2_x, trace2_y = [], []

    def update(frame):
        idx = frame * t_per_frame
        if idx >= len(t):
            idx = len(t) - 1

        x1_t, y1_t = x1[idx], y1[idx]
        x2_t, y2_t = x2[idx], y2[idx]

        # Bodies
        body1.set_data([x1_t], [y1_t])
        body2.set_data([x2_t], [y2_t])

        # Connection
        connection.set_data([x1_t, x2_t], [y1_t, y2_t])

        # Traces
        trace1_x.append(x1_t)
        trace1_y.append(y1_t)
        trace2_x.append(x2_t)
        trace2_y.append(y2_t)
        trace1.set_data(trace1_x, trace1_y)
        trace2.set_data(trace2_x, trace2_y)

        # Time
        time_text.set_text(f'Time: {t[idx]:.2f} s')

        return body1, body2, trace1, trace2, connection, time_text

    num_frames = max(1, len(t) // t_per_frame)
    ani = FuncAnimation(fig, update, frames=num_frames, blit=True, interval=50)
    # Option A: Save as GIF (requires pillow installed)
    ani.save("orbit_animation.gif", writer="pillow", fps=20)

    plt.close(fig)  # prevent duplicate static image
    return HTML(ani.to_jshtml())

In [ ]:
visualize(data, cond, traj_index=0, num_points=None, make_animation=True, t_per_frame=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def visualize(
    data,
    cond,
    traj_index=0,
    num_points=None,
    make_animation=True,
    t_per_frame=1,
):
    """
    Visualize one two-body trajectory (GT vs prediction).
    """

    # ---------------------------------------------------------
    # Slice trajectory
    # ---------------------------------------------------------
    t_full = data['t'][traj_index]
    if num_points is None:
        num_points = t_full.shape[0]

    t      = t_full[:num_points]
    q_gt   = data['q_gt'][traj_index][:num_points]
    q_pred = data['q_pred'][traj_index][:num_points]
    p_gt   = data['p_gt'][traj_index][:num_points]
    p_pred = data['p_pred'][traj_index][:num_points]

    r_gt,    theta_gt    = q_gt[:, 0], q_gt[:, 1]
    r_pred,  theta_pred  = q_pred[:, 0], q_pred[:, 1]
    dr_gt,   dtheta_gt   = p_gt[:, 0], p_gt[:, 1]
    dr_pred, dtheta_pred = p_pred[:, 0], p_pred[:, 1]

    # Wrap theta to [0, 2π] for visualization
    theta_wrapped_gt   = theta_gt   % (2 * np.pi)
    theta_wrapped_pred = theta_pred % (2 * np.pi)

    # ---------------------------------------------------------
    # Physical parameters
    # ---------------------------------------------------------
    m1 = float(cond['m1'][traj_index])
    m2 = float(cond['m2'][traj_index])
    M  = m1 + m2
    G  = 1.0

    # ---------------------------------------------------------
    # 1) q–t curves (r, θ vs time)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, r_gt,              label='r (radial) – GT',   color='C0')
    plt.plot(t, r_pred,            label='r (radial) – pred', color='C1')
    plt.plot(t, theta_wrapped_gt,  label='θ (angle) – GT',    color='C0', linestyle='--')
    plt.plot(t, theta_wrapped_pred,label='θ (angle) – pred',  color='C1', linestyle='--')
    plt.title('Positions vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Position [r: distance, θ: rad]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 2) dq–t curves (velocities vs time)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(t, dr_gt,       label='dr/dt – GT',   color='C0')
    plt.plot(t, dr_pred,     label='dr/dt – pred', color='C1')
    plt.plot(t, dtheta_gt,   label='dθ/dt – GT',   color='C0', linestyle='--')
    plt.plot(t, dtheta_pred, label='dθ/dt – pred', color='C1', linestyle='--')
    plt.title('Velocities vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 3) Phase space
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(r_gt,       dr_gt,       label='r vs dr/dt – GT',   color='C0', alpha=0.7)
    plt.plot(r_pred,     dr_pred,     label='r vs dr/dt – pred', color='C1', alpha=0.7)
    plt.plot(theta_wrapped_gt,   dtheta_gt,   label='θ vs dθ/dt – GT',   color='C0', linestyle='--', alpha=0.7)
    plt.plot(theta_wrapped_pred, dtheta_pred, label='θ vs dθ/dt – pred', color='C1', linestyle='--', alpha=0.7)
    plt.title('Phase Space')
    plt.xlabel('Position [r: distance, θ: rad]')
    plt.ylabel('Velocity [dr/dt: distance/s, dθ/dt: rad/s]')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 4) Orbit trajectories (2D, GT vs pred) calculation
    # ---------------------------------------------------------
    # Reduced coordinate → Cartesian
    x_rel_gt   = r_gt   * np.cos(theta_gt)
    y_rel_gt   = r_gt   * np.sin(theta_gt)
    x_rel_pred = r_pred * np.cos(theta_pred)
    y_rel_pred = r_pred * np.sin(theta_pred)

    # Positions of bodies in CoM frame (GT)
    x1_gt = -m2 / M * x_rel_gt
    y1_gt = -m2 / M * y_rel_gt
    x2_gt =  m1 / M * x_rel_gt
    y2_gt =  m1 / M * y_rel_gt

    # Positions of bodies in CoM frame (Pred)
    x1_pred = -m2 / M * x_rel_pred
    y1_pred = -m2 / M * y_rel_pred
    x2_pred =  m1 / M * x_rel_pred
    y2_pred =  m1 / M * y_rel_pred

    plt.figure(figsize=(8, 8), dpi=100)
    # GT
    plt.plot(x1_gt, y1_gt, '-',  label='Body 1 – GT',   color='C0', alpha=0.7)
    plt.plot(x2_gt, y2_gt, '--', label='Body 2 – GT',   color='C0', alpha=0.7)
    # Pred
    plt.plot(x1_pred, y1_pred, '-',  label='Body 1 – pred', color='C1', alpha=0.7)
    plt.plot(x2_pred, y2_pred, '--', label='Body 2 – pred', color='C1', alpha=0.7)

    # Start/end markers (GT)
    plt.plot(x1_gt[0],  y1_gt[0],  'o', color='C0', label='Body 1 start (GT)')
    plt.plot(x2_gt[0],  y2_gt[0],  's', color='C0', label='Body 2 start (GT)')
    plt.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='Center of Mass')

    plt.title(f'Orbit Trajectories (GT vs Pred)\n m1={m1:.2f}, m2={m2:.2f}')
    plt.xlabel('x position')
    plt.ylabel('y position')
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 5) Animated orbit (GT + Pred)
    # ---------------------------------------------------------
    if not make_animation:
        return

    # Determine plot limits based on both GT and Pred
    all_x = np.concatenate([x1_gt, x2_gt, x1_pred, x2_pred])
    all_y = np.concatenate([y1_gt, y2_gt, y1_pred, y2_pred])
    r_max = np.max(np.sqrt(all_x**2 + all_y**2))
    plot_margin = 0.2 * r_max
    xlim = (-r_max - plot_margin, r_max + plot_margin)
    ylim = (-r_max - plot_margin, r_max + plot_margin)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_title(f'Two-Body Orbit Animation (GT vs Pred)\nBlue=GT, Orange=Pred')
    ax.set_xlabel('x position')
    ax.set_ylabel('y position')
    ax.grid(True, alpha=0.3)

    # CoM marker
    ax.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='CoM')

    # --- GT Objects (Blue C0) ---
    body1_gt, = ax.plot([], [], 'o', color='C0', markersize=8, label='Body 1 (GT)')
    body2_gt, = ax.plot([], [], 's', color='C0', markersize=8, label='Body 2 (GT)')
    trace1_gt, = ax.plot([], [], '-', color='C0', alpha=0.4)
    trace2_gt, = ax.plot([], [], '--', color='C0', alpha=0.4)
    conn_gt,   = ax.plot([], [], '-', color='gray', alpha=0.3, linewidth=1)

    # --- Pred Objects (Orange C1) ---
    body1_pred, = ax.plot([], [], 'o', color='C1', markersize=8, label='Body 1 (Pred)')
    body2_pred, = ax.plot([], [], 's', color='C1', markersize=8, label='Body 2 (Pred)')
    trace1_pred, = ax.plot([], [], '-', color='C1', alpha=0.4)
    trace2_pred, = ax.plot([], [], '--', color='C1', alpha=0.4)

    # Time label
    time_text = ax.text(
        0.02, 0.98, '', transform=ax.transAxes, va='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )

    ax.legend(loc='upper right', fontsize='small', framealpha=0.8)

    # History lists for traces
    t1_gt_x, t1_gt_y = [], []
    t2_gt_x, t2_gt_y = [], []
    t1_pr_x, t1_pr_y = [], []
    t2_pr_x, t2_pr_y = [], []

    def update(frame):
        idx = frame * t_per_frame
        if idx >= len(t):
            idx = len(t) - 1

        # Current coords
        x1g, y1g = x1_gt[idx], y1_gt[idx]
        x2g, y2g = x2_gt[idx], y2_gt[idx]
        x1p, y1p = x1_pred[idx], y1_pred[idx]
        x2p, y2p = x2_pred[idx], y2_pred[idx]

        # Update Bodies
        body1_gt.set_data([x1g], [y1g])
        body2_gt.set_data([x2g], [y2g])
        body1_pred.set_data([x1p], [y1p])
        body2_pred.set_data([x2p], [y2p])

        # Update Connection (GT only)
        conn_gt.set_data([x1g, x2g], [y1g, y2g])

        # Update Traces
        t1_gt_x.append(x1g); t1_gt_y.append(y1g)
        t2_gt_x.append(x2g); t2_gt_y.append(y2g)
        t1_pr_x.append(x1p); t1_pr_y.append(y1p)
        t2_pr_x.append(x2p); t2_pr_y.append(y2p)

        trace1_gt.set_data(t1_gt_x, t1_gt_y)
        trace2_gt.set_data(t2_gt_x, t2_gt_y)
        trace1_pred.set_data(t1_pr_x, t1_pr_y)
        trace2_pred.set_data(t2_pr_x, t2_pr_y)

        # Update Time
        time_text.set_text(f'Time: {t[idx]:.2f} s')

        return (body1_gt, body2_gt, trace1_gt, trace2_gt, conn_gt,
                body1_pred, body2_pred, trace1_pred, trace2_pred, time_text)

    num_frames = max(1, len(t) // t_per_frame)
    ani = FuncAnimation(fig, update, frames=num_frames, blit=True, interval=50)

    ani.save("orbit_animation.gif", writer="pillow", fps=20)
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [ ]:
visualize(data, cond, traj_index=2, num_points=None, make_animation=True, t_per_frame=1)

### Reproducing the paper plots

In [ ]:
# path = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise1_update1/result_dict.npy'
path = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise5_update1/result_dict.npy'
path_cond = '/root/work/dhn-deeplearning/results/ar/two_body_kernel2_stride1/gen_sequence/denoise5_update1/cond_dict.npy'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

data = np.load(path, allow_pickle=True).item()
cond = np.load(path_cond, allow_pickle=True).item()
print(data.keys())
print(cond.keys())

In [ ]:
# Let's first do the average error on q over time plot

t=data['t']
q_gt=data['q_gt']
q_pred=data['q_pred']

r_gt=q_gt[:,:,0]
r_pred=q_pred[:,:,0]
theta_gt=q_gt[:,:,1]
theta_pred=q_pred[:,:,1]

residual_r=np.abs(r_gt-r_pred)
residual_theta=np.abs(theta_gt-theta_pred)

plt.plot(t[0], residual_r.mean(axis=0), label='r')
plt.plot(t[0], residual_theta.mean(axis=0), label='theta')
plt.xlabel('Time [s]')
plt.ylabel('Average Error')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Now let's calculate the energy over time

m1 = cond['m1']
m2 = cond['m2']
mu = m1*m2/(m1+m2)
M = m1+m2
G = 1.0

q_gt=data['q_gt']
q_pred=data['q_pred']

r_gt=q_gt[:,:,0]
r_pred=q_pred[:,:,0]
theta_gt=q_gt[:,:,1]
theta_pred=q_pred[:,:,1]

dr_gt = r_gt[:,:-1] - r_gt[:,1:]
dr_pred = r_pred[:,:-1] - r_pred[:,1:]

dtheta_gt = theta_gt[:,:-1] - theta_gt[:,1:]
dtheta_pred = theta_pred[:,:-1] - theta_pred[:,1:]


# T = mu/2 * (dr**2 + r**2 * dtheta**2)
# V = -G*M/r

T_gt = mu.reshape(-1,1)/2 * (dr_gt**2 + (r_gt[:,:-1])**2 * dtheta_gt**2)
T_pred = mu.reshape(-1,1)/2 * (dr_pred**2 + (r_pred[:,:-1])**2 * dtheta_pred**2)

V_gt = -(G*M).reshape(-1,1)/r_gt[:,:-1]
V_pred = -(G*M).reshape(-1,1)/r_pred[:,:-1]

E_gt = T_gt + V_gt
E_pred = T_pred + V_pred

plt.plot(t[0][:-1], E_gt.mean(axis=0), label='GT')
plt.plot(t[0][:-1], E_pred.mean(axis=0), label='Pred')
plt.xlabel('Time [s]')
plt.ylabel('Energy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Let's start by looking at a single trajectory

traj_index = 0

q_gt = data['q_gt'][traj_index]
q_pred = data['q_pred'][traj_index]

r_gt = q_gt[:,0]
r_pred = q_pred[:,0]
theta_gt = q_gt[:,1]
theta_pred = q_pred[:,1]

dt = t[0][1] - t[0][0]

dr_gt = (r_gt[1:] - r_gt[:-1])/dt
dr_pred = (r_pred[1:] - r_pred[:-1])/dt

dtheta_gt = theta_gt[1:] - theta_gt[:-1]
dtheta_pred = theta_pred[1:] - theta_pred[:-1]


# T = mu/2 * (dr**2 + r**2 * dtheta**2)
# V = -G*M/r

T_gt = mu.reshape(-1,1)/2 * (dr_gt**2 + (r_gt[:-1])**2 * dtheta_gt**2)
T_pred = mu.reshape(-1,1)/2 * (dr_pred**2 + (r_pred[:-1])**2 * dtheta_pred**2)

V_gt = -(G*m1*m2).reshape(-1,1)/r_gt[:-1]
V_pred = -(G*m1*m2).reshape(-1,1)/r_pred[:-1]

E_gt = T_gt + V_gt
E_pred = T_pred + V_pred

plt.plot(t[0][:-1], E_gt.mean(axis=0), label='GT')
plt.plot(t[0][:-1], E_pred.mean(axis=0), label='Pred')
plt.xlabel('Time [s]')
plt.ylabel('Energy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
m1 = cond['m1']   # shape (B,)
m2 = cond['m2']   # shape (B,)
mu = m1 * m2 / (m1 + m2)   # shape (B,)
G = 1.0

q_gt = data['q_gt']   # (B, T, 2)
q_pred = data['q_pred']

r_gt = q_gt[:, :, 0]      # (B, T)
r_pred = q_pred[:, :, 0]
theta_gt = q_gt[:, :, 1]
theta_pred = q_pred[:, :, 1]

# # unwrap angles to avoid 2π jumps
# theta_gt = np.unwrap(theta_gt, axis=1)
# theta_pred = np.unwrap(theta_pred, axis=1)

# time differences (assume same t for all trajectories)
t = data['t']          # (B, T) or (1, T)
if t.ndim == 2:
    t0 = t[0]
else:
    t0 = t
dt = t0[1:] - t0[:-1]     # (T-1,)

# finite differences for velocities: forward difference / dt
dr_gt = (r_gt[:, 1:] - r_gt[:, :-1]) / dt[None, :]        # (B, T-1)
dr_pred = (r_pred[:, 1:] - r_pred[:, :-1]) / dt[None, :]

dtheta_gt = (theta_gt[:, 1:] - theta_gt[:, :-1]) / dt[None, :]
dtheta_pred = (theta_pred[:, 1:] - theta_pred[:, :-1]) / dt[None, :]

# use r at the midpoints (same indexing as velocities)
r_gt_mid = r_gt[:, 1:]
r_pred_mid = r_pred[:, 1:]

# masses
mu_col = mu.reshape(-1, 1)          # (B, 1)
mprod_col = (m1 * m2).reshape(-1, 1)

# Kinetic: T = 0.5 * mu * (dr^2 + r^2 dtheta^2)
T_gt = 0.5 * mu_col * (dr_gt**2 + r_gt_mid**2 * dtheta_gt**2)
T_pred = 0.5 * mu_col * (dr_pred**2 + r_pred_mid**2 * dtheta_pred**2)

# Potential: V = -G * m1 * m2 / r
V_gt = - G * mprod_col / r_gt_mid
V_pred = - G * mprod_col / r_pred_mid

E_gt = T_gt + V_gt        # (B, T-1)
E_pred = T_pred + V_pred

# Plot mean energy over trajectories
plt.plot(t0[1:], E_gt.mean(axis=0), label='GT')
plt.plot(t0[1:], E_pred.mean(axis=0), label='Pred')
plt.xlabel('Time [s]')
plt.ylabel('Energy')
plt.legend()
plt.grid(True)
plt.show()